In [ ]:
# General imports
import anthropic
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic.types import ToolParam
from datetime import datetime

def get_current_datetime(format="%Y-%m-%d %H:%M:%S"):
    if not format:
        raise ValueError("Format string cannot be empty")
    return datetime.now().strftime(format)

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Retrieves the current date and time in the specified format. This tool is useful when you need to know the present date, time, or both. You can customize the output format using Python strftime syntax (e.g., '%Y-%m-%d' for just the date, '%H:%M:%S' for just the time). If no format is provided, it defaults to 'YYYY-MM-DD HH:MM:SS' format. The tool will raise an error if an empty format string is provided.",
        "input_schema": {
            "type": "object",
            "properties": {
            "format": {
                "type": "string",
                "description": "Python strftime format string to control the output format. Common examples: '%Y-%m-%d' (2024-05-27), '%H:%M:%S' (14:30:45), '%Y-%m-%d %H:%M:%S' (2024-05-27 14:30:45), '%A, %B %d, %Y' (Monday, May 27, 2024). Defaults to '%Y-%m-%d %H:%M:%S' if not provided. Cannot be an empty string.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
            },
            "required": []
        },
        "input_examples": [
            {
            "format": "%Y-%m-%d"
            },
            {
            "format": "%H:%M:%S"
            },
            {
            "format": "%A, %B %d, %Y at %I:%M %p"
            },
            {}
        ]
    }
)     

In [ ]:
from anthropic.types import Message

def add_user_messages(messages, message):
    messages.append(
        {
            "role": "user", 
            "content": message.content if isinstance(message, Message) else message
        }
    )

def add_assistant_messages(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message
        }
    )

    
def chat_stream(messages, stop_sequences=[], temperature=1.0, system=None, tools=None):
    client = anthropic.Anthropic()
    parameters = {
        "model": "claude-haiku-4-5",
        "max_tokens": 1024 * 10,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if tools:
        parameters["tools"] = tools

    if system:
        parameters["system"] = system

    # with client.messages.stream(**parameters) as stream:
        # for chunk in stream: 
            # if chunk.type == "text": 
                # print(chunk.text, end="")
            # print(chunk)/
    return client.messages.stream(**parameters)

In [ ]:
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [ ]:
import json

text_editor_tool = TextEditorTool()

def run_tool(tool_request): 
    try:
        tool_output = ""
        if(tool_request.name == "get_current_datetime"):
            tool_output = get_current_datetime(**tool_request.input)
        if tool_request.name == "str_replace_based_edit_tool":
            command = tool_request.input["command"]
            if command == "view":
                tool_output = text_editor_tool.view(
                    tool_request.input["path"], tool_request.input.get("view_range")
                )
            elif command == "str_replace":
                tool_output = text_editor_tool.str_replace(
                    tool_request.input["path"], tool_request.input["old_str"], tool_request.input["new_str"]
                )
            elif command == "create":
                tool_output = text_editor_tool.create(tool_request.input["path"], tool_request.input["file_text"])
            elif command == "insert":
                tool_output = text_editor_tool.insert(
                    tool_request.input["path"],
                    tool_request.input["insert_line"],
                    tool_request.input["new_str"],
                )
            elif command == "undo_edit":
                tool_output = text_editor_tool.undo_edit(tool_request.input["path"])
        # if(tool_request.name == "add_duration_to_datetime"):
            # tool_output = add_duration_to_datetime(**tool_request.input)
        # if(tool_request.name == "set_reminder"):
            # tool_output = set_reminder(**tool_request.input)
        if(tool_output == ""):
            raise ValueError("tool output is empty")
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False
        }
    except Exception as e:
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": f"Error: {e}",
            "is_error": True
        }
def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_responses = []

    for tool_request in tool_requests:
        tool_responses.append(run_tool(tool_request))

    return tool_responses

In [ ]:
messages = []
first_message = "Read file `./main.py` and summarize content. After: Open the ./main.py file and write out a function to calculate pi to the 5th digit. Then create a ./test.py file to test your implementation."
print(first_message)
add_user_messages(messages, first_message)
# add_user_messages(messages,"Write one sentence")
while True:
    with chat_stream(messages, tools=[get_current_datetime_schema, {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }]) as stream:
        for chunk in stream:
            if chunk.type == "text":
                print(chunk.text, end="")

            if chunk.type == "content_block_start":
                if chunk.content_block.type == "tool_use":
                    print(f'\n>>> Tool Call: "{chunk.content_block.name}"')
            
            if chunk.type == "content_block_stop":
                print("\n")
        
        response = stream.get_final_message()
        print(response)
        add_assistant_messages(messages, response)

        if(response.stop_reason != "tool_use"): 
            break

        tools_response = run_tools(response)
        add_user_messages(messages, tools_response)
messages